# Sta-RU Video Dubbing — Edge-TTS variant

Same pipeline as the main notebook but uses **Microsoft Edge's neural TTS** (free, no API key, no voice cloning). Much faster (~30s per video instead of 6 min) and gives a stable, broadcast-grade voice.

Use this notebook when you don't need to preserve the original speaker's timbre and prefer a clean stock narrator.

## 1. GPU check (optional)

GPU is only used by Demucs. The TTS itself runs on CPU and is plenty fast.

In [ ]:
!nvidia-smi | head -20

## 2. Install dependencies

In [ ]:
!apt-get -qq install -y ffmpeg rubberband-cli
!pip install -q edge-tts nest-asyncio yt-dlp srt soundfile numpy scipy demucs deep-translator pyrubberband ipywidgets pandas

## 3. Mount Google Drive (optional)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 4. Clone the Sta-RU repo

In [ ]:
import os, sys
REPO_DIR = '/content/Sta-RU'
BRANCH = 'claude/compassionate-knuth-M2llL'
if not os.path.exists(REPO_DIR):
    !git clone --depth 1 -b $BRANCH https://github.com/lazy-money/sta-ru.git $REPO_DIR
else:
    !cd $REPO_DIR && git pull --quiet
sys.path.insert(0, os.path.join(REPO_DIR, 'colab'))
print('Repo ready at', REPO_DIR, '(branch:', BRANCH, ')')

## 5. Provide YouTube URLs

One per line. Use `2: https://...` to force a specific N#.

In [ ]:
import ipywidgets as W
from IPython.display import display

url_box = W.Textarea(
    value='',
    placeholder=('Paste one YouTube URL per line.\n'
                 'https://www.youtube.com/watch?v=...\n'
                 '\nOr force N#:\n'
                 '2: https://www.youtube.com/watch?v=...'),
    description='URLs:',
    layout=W.Layout(width='100%', height='180px'),
)
display(url_box)

## 6. Upload subtitle files

Format: `{N#}-{LANG}.srt` (e.g. `1-EN.srt`, `47-ES.srt`). Multi-select supported.

In [ ]:
from google.colab import files
import os, re

SRT_DIR = '/content/srts'
os.makedirs(SRT_DIR, exist_ok=True)
_srts = files.upload()
_pattern = re.compile(r'^(\d+)-([A-Z]{2})\.srt$')
loaded = []
for name, content in _srts.items():
    with open(os.path.join(SRT_DIR, name), 'wb') as f:
        f.write(content)
    m = _pattern.match(name)
    if m:
        loaded.append((int(m.group(1)), m.group(2)))
print(f'\nLoaded {len(loaded)} SRTs')
by_lang = {}
for n, lg in loaded:
    by_lang.setdefault(lg, []).append(n)
for lg, ns in by_lang.items():
    print(f'  {lg}: N# {sorted(ns)}')

## 7. Options

In [ ]:
import ipywidgets as W
from IPython.display import display
from batch_dub_edge import DEFAULT_VOICES

LANG_OPTIONS = list(DEFAULT_VOICES.keys())

lang_w            = W.Dropdown(options=LANG_OPTIONS, value='EN', description='Target language:')
gender_w          = W.RadioButtons(options=[('Male', 'M'), ('Female', 'F')], value='M', description='Voice gender:')
voice_custom_w    = W.Text(value='', placeholder='Optional: override the default voice (e.g. en-US-BrianNeural)',
                           description='Custom voice:', layout=W.Layout(width='600px'))
pitch_w           = W.IntSlider(value=-5, min=-20, max=20, step=1, description='Pitch (Hz):')
range_w           = W.Text(value='all', description='Range (N#):', placeholder="'all', '1-10', '47'")
remove_voice_w    = W.Checkbox(value=True,  description='Remove original voice (Demucs)')
dynamic_dur_w     = W.Checkbox(value=False, description='Dynamic duration (stretch video to fit dubbing)')
skip_silent_w     = W.Checkbox(value=True,  description='Skip TTS where the original speaker is silent')
burn_subs_w       = W.Checkbox(value=False, description='Burn subtitles into the video')
translate_title_w = W.Checkbox(value=False, description='Translate title to target language')

out_mode_w = W.RadioButtons(
    options=[('Save to Google Drive', 'drive'), ('Save locally', 'local')],
    value='drive', description='Output:',
)
out_path_w = W.Text(value='/content/drive/MyDrive/Dubbing/EN', description='Output dir:',
                    layout=W.Layout(width='600px'))
cache_path_w = W.Text(value='/tmp/sta-ru-cache', description='Cache dir:',
                      layout=W.Layout(width='600px'))

def _sync(_=None):
    lg = lang_w.value
    out_path_w.value = (f'/content/drive/MyDrive/Dubbing/{lg}' if out_mode_w.value == 'drive'
                        else f'/content/output/{lg}')
lang_w.observe(_sync, names='value')
out_mode_w.observe(_sync, names='value')

display(lang_w, gender_w, voice_custom_w, pitch_w, range_w,
        remove_voice_w, dynamic_dur_w, skip_silent_w, burn_subs_w,
        translate_title_w, out_mode_w, out_path_w, cache_path_w)
print()
print('Dynamic duration: voice stays natural; video stretches to match. ~3-5x slower per video.')
print('Skip silent: avoids dubbing over moments the speaker stayed quiet (Whisper hallucinations).')
print('Burn subtitles: bakes the SRT into the video (forces re-encode, slower).')
print('Cache dir: persists video/ambient per URL so other languages reuse them.')
print()
print('Adjust the values above, then run the next cell.')

In [ ]:
# Lock in configuration
CONFIG = {
    'lang':                 lang_w.value,
    'gender':               gender_w.value,
    'voice':                voice_custom_w.value.strip() or None,
    'pitch_st':             pitch_w.value,
    'range_expr':           range_w.value,
    'remove_voice':         remove_voice_w.value,
    'dynamic_duration':     dynamic_dur_w.value,
    'skip_silent_segments': skip_silent_w.value,
    'burn_in_subs':         burn_subs_w.value,
    'translate_titles':     translate_title_w.value,
    'output_dir':           out_path_w.value,
    'srt_dir':              SRT_DIR,
    'cache_root':           cache_path_w.value or None,
}

url_text = (url_box.value or '').strip()
if not url_text:
    raise RuntimeError('No URLs provided. Paste in the textarea.')
URL_SOURCE = [l for l in url_text.splitlines() if l.strip()]
print(f'URL source: textarea ({len(URL_SOURCE)} lines)')

from batch_dub_edge import resolve_voice
effective_voice = resolve_voice(CONFIG['lang'], CONFIG['gender'], CONFIG['voice'])
print(f'Effective voice: {effective_voice}')
print('\nConfiguration:')
for k, v in CONFIG.items():
    print(f'  {k:22} = {v}')

## 8. Preview

In [ ]:
import pandas as pd
from batch_dub import load_urls, build_items, build_output_name

all_urls = load_urls(URL_SOURCE)
preview_urls = all_urls[:3]
preview_items = build_items(preview_urls,
                            translate_titles=CONFIG['translate_titles'],
                            target_lang=CONFIG['lang'].lower())
rows = []
for it in preview_items:
    rows.append({
        'N#': it.n,
        'Date': it.upload_date or '—',
        'Duration': f'{int(it.duration//60)}:{int(it.duration%60):02d}' if it.duration else '—',
        'Title': it.title or f'[fallback — {it.error}]',
        'Output filename': build_output_name(it, CONFIG['lang'], CONFIG['translate_titles']),
    })
pd.DataFrame(rows).style.set_properties(**{'text-align': 'left'}).hide(axis='index')

## 9. Run batch

In [ ]:
from batch_dub_edge import run_batch

results = run_batch(
    urls=URL_SOURCE,
    srt_dir=CONFIG['srt_dir'],
    output_dir=CONFIG['output_dir'],
    lang=CONFIG['lang'],
    gender=CONFIG['gender'],
    voice=CONFIG['voice'],
    pitch_st=CONFIG['pitch_st'],
    translate_titles=CONFIG['translate_titles'],
    remove_voice=CONFIG['remove_voice'],
    dynamic_duration=CONFIG['dynamic_duration'],
    skip_silent_segments=CONFIG['skip_silent_segments'],
    burn_in_subs=CONFIG['burn_in_subs'],
    cache_root=CONFIG['cache_root'],
    range_expr=CONFIG['range_expr'],
)

## 10. Download outputs (local mode only)

In [ ]:
from google.colab import files
from pathlib import Path
for it in results:
    if it.status == 'done' and it.output_path and 'drive' not in str(it.output_path):
        files.download(str(it.output_path))

## Reset for next run

Wipes the cached Python modules from this session **and** the cloned repo on disk. Run this when you want the next iteration to pick up new code (after a `git pull` on the repo).

In [ ]:
# Wipe everything that gets stale between runs:
#   - cached batch_dub modules in this Python session
#   - cloned repo on disk (forces a fresh git clone next time you run cell 4)
# Run this when you want the next iteration to pick up new code.
import sys
for _m in list(sys.modules):
    if _m.startswith('batch_dub'):
        del sys.modules[_m]
print('Cleared cached batch_dub modules')

!rm -rf /content/Sta-RU
print('Removed /content/Sta-RU')